# HEDIS Analytics — Exploratory Notebook
Phase 4 verification and Phase 5 data exploration.
Connect to SQL Server and run test queries against the loaded CMS data.

In [2]:
from sqlalchemy import create_engine
import pandas as pd
import pyodbc
import getpass
from urllib.parse import quote_plus

password = getpass.getpass('SA password: ')

conn_str = (
    'DRIVER={ODBC Driver 18 for SQL Server};'
    'SERVER=localhost;'
    'DATABASE=hedis;'
    'UID=SA;'
    f'PWD={password};'
    'TrustServerCertificate=yes;'
)

engine = create_engine(f'mssql+pyodbc:///?odbc_connect={quote_plus(conn_str)}')
conn = engine.connect()
print('Connected.')

SA password:  ········


Connected.


## Row counts — verify all tables loaded

In [2]:
tables = ['beneficiary', 'inpatient', 'outpatient', 'carrier', 'dme', 'hha', 'hospice', 'snf', 'pde']

counts = []
for t in tables:
    df = pd.read_sql(f'SELECT COUNT(*) AS row_count FROM {t}', conn)
    counts.append({'table': t, 'row_count': df['row_count'][0]})

pd.DataFrame(counts)

,table,row_count
0,beneficiary,10000
1,inpatient,58066
2,outpatient,575092
3,carrier,1121004
4,dme,103828
5,hha,6215
6,hospice,12107
7,snf,12548
8,pde,515520


## Beneficiary — sample rows

In [3]:
pd.read_sql("""
    SELECT TOP 5
        BENE_ID,
        BENE_BIRTH_DT,
        AGE_AT_END_REF_YR,
        SEX_IDENT_CD,
        BENE_RACE_CD,
        BENE_DEATH_DT,
        BENE_ENROLLMT_REF_YR,
        STATE_CODE,
        ZIP_CD
    FROM beneficiary
""", conn)

,BENE_ID,BENE_BIRTH_DT,AGE_AT_END_REF_YR,SEX_IDENT_CD,BENE_RACE_CD,BENE_DEATH_DT,BENE_ENROLLMT_REF_YR,STATE_CODE,ZIP_CD
0,-10000010254618,16-Aug-1999,26,1,1,None,2025,01,36109
1,-10000010254647,14-Aug-1990,35,1,2,None,2025,01,35756
2,-10000010254653,18-Mar-1982,43,2,1,None,2025,01,36801
3,-10000010254655,17-Mar-1958,67,2,2,None,2025,01,35071
4,-10000010254656,24-Jul-1999,26,2,1,None,2025,01,35216


## Beneficiary — age and sex distribution

In [4]:
pd.read_sql("""
    SELECT
        SEX_IDENT_CD,
        MIN(AGE_AT_END_REF_YR) AS min_age,
        MAX(AGE_AT_END_REF_YR) AS max_age,
        AVG(CAST(AGE_AT_END_REF_YR AS FLOAT)) AS avg_age,
        COUNT(*) AS bene_count
    FROM beneficiary
    GROUP BY SEX_IDENT_CD
    ORDER BY SEX_IDENT_CD
""", conn)

,SEX_IDENT_CD,min_age,max_age,avg_age,bene_count
0,1,2,113,62.979175,4994
1,2,2,113,63.329804,5006


## Claims — date range check

In [5]:
pd.read_sql("""
    SELECT 'inpatient' AS claim_type, MIN(CLM_FROM_DT) AS earliest, MAX(CLM_THRU_DT) AS latest FROM inpatient
    UNION ALL
    SELECT 'outpatient', MIN(CLM_FROM_DT), MAX(CLM_THRU_DT) FROM outpatient
    UNION ALL
    SELECT 'carrier', MIN(CLM_FROM_DT), MAX(CLM_THRU_DT) FROM carrier
    UNION ALL
    SELECT 'pde', MIN(SRVC_DT), MAX(SRVC_DT) FROM pde
""", conn)

,claim_type,earliest,latest
0,inpatient,01-Apr-2015,31-Oct-2022
1,outpatient,01-Apr-2015,31-Oct-2022
2,carrier,01-Apr-2015,31-Oct-2022
3,pde,01-Apr-2015,31-Oct-2022


## Inpatient — sample diagnosis codes

In [6]:
pd.read_sql("""
    SELECT TOP 10
        BENE_ID,
        CLM_FROM_DT,
        CLM_THRU_DT,
        PRNCPAL_DGNS_CD,
        ICD_DGNS_CD1,
        ICD_DGNS_CD2,
        ICD_DGNS_CD3
    FROM inpatient
    WHERE CLM_LINE_NUM = 1
""", conn)

,BENE_ID,CLM_FROM_DT,CLM_THRU_DT,PRNCPAL_DGNS_CD,ICD_DGNS_CD1,ICD_DGNS_CD2,ICD_DGNS_CD3
0,-10000010254618,25-Mar-2015,25-Mar-2015,S134XX,S134XX,R4689,E781
1,-10000010254653,24-Sep-2015,24-Sep-2015,Z3480,T7432X,E669,C50919
2,-10000010254653,09-May-2017,10-May-2017,T7432X,T7432X,E669,C50929
3,-10000010254656,14-Jan-2017,14-Jan-2017,S8290X,S8290X,G40909,R569
4,-10000010254656,17-Mar-2018,17-Mar-2018,Z3480,Z5989,Z5941,G40909
5,-10000010254656,12-Mar-2022,12-Mar-2022,Z3400,T7432X,Z733,Z5989
6,-10000010254666,21-Mar-2015,21-Mar-2015,R569,G40909,P90,Z8669
7,-10000010254682,30-Apr-2021,30-Apr-2021,S8290X,S8290X,Z733,I252
8,-10000010254691,12-Mar-2020,12-Mar-2020,Z608,Z608,Z7682,N184
9,-10000010254691,21-May-2020,21-May-2020,Z608,Z608,Z7682,N184


## Carrier — sample HCPCS codes

In [7]:
pd.read_sql("""
    SELECT TOP 10
        BENE_ID,
        CLM_FROM_DT,
        HCPCS_CD,
        LINE_ICD_DGNS_CD,
        LINE_NCH_PMT_AMT
    FROM carrier
    WHERE HCPCS_CD IS NOT NULL
""", conn)

,BENE_ID,CLM_FROM_DT,HCPCS_CD,LINE_ICD_DGNS_CD,LINE_NCH_PMT_AMT
0,-10000010254618,28-Sep-2015,96127,None,367.28
1,-10000010254618,28-Sep-2015,G0444,R4689,0.00
2,-10000010254618,28-Sep-2015,G9573,None,0.00
3,-10000010254618,03-Oct-2016,96127,R4689,367.28
4,-10000010254618,03-Oct-2016,G0444,None,0.00
5,-10000010254618,03-Oct-2016,G9573,R4689,0.00
6,-10000010254618,03-Oct-2016,99408,R4689,0.00
7,-10000010254618,09-Oct-2017,99495,R4689,302.24
8,-10000010254618,09-Oct-2017,96156,None,367.28
9,-10000010254618,09-Oct-2017,96127,None,0.00


## PDE — sample drug records

In [8]:
pd.read_sql("""
    SELECT TOP 10
        BENE_ID,
        SRVC_DT,
        PROD_SRVC_ID,
        DAYS_SUPLY_NUM,
        TOT_RX_CST_AMT,
        DRUG_CVRG_STUS_CD
    FROM pde
""", conn)

,BENE_ID,SRVC_DT,PROD_SRVC_ID,DAYS_SUPLY_NUM,TOT_RX_CST_AMT,DRUG_CVRG_STUS_CD
0,-10000010254618,25-Mar-2015,68115025030,63,275.19,C
1,-10000010254618,27-May-2016,53978010903,7,2420.67,C
2,-10000010254618,03-Oct-2016,55154010000,90,1204.91,C
3,-10000010254618,20-Sep-2017,13107021199,10,136.48,C
4,-10000010254618,30-Sep-2017,13107021199,10,136.48,C
5,-10000010254618,09-Oct-2017,11014070601,90,1388.21,C
6,-10000010254618,09-Oct-2017,10702019250,10,104.13,C
7,-10000010254618,19-Oct-2017,10702019250,10,104.13,C
8,-10000010254618,29-Oct-2017,10702019250,10,104.13,C
9,-10000010254618,08-Nov-2017,10702019250,10,104.13,C


---
## Phase 5 — Measure Coverage Review
Measurement year: 2021 (Jan 1 – Dec 31, 2021).
Age calculated from BENE_BIRTH_DT as of Dec 31, 2021.
Goal: confirm eligible population size and code presence for each candidate measure.

### Query 1 — Eligible population by measure

Every HEDIS measure has a denominator — the population who is eligible to be measured. Age and sex are the first filter. For example, BCS (breast cancer screening) only applies to females aged 52-74. COL (colorectal cancer screening) applies to everyone aged 45-75.

This query calculates each beneficiary's age as of Dec 31, 2021 from their birth date, then counts how many fall into each measure's age/sex window. This tells us if we have enough people in the data to produce a meaningful rate for each measure.

In [3]:
pd.read_sql("""
    WITH aged AS (
        SELECT
            BENE_ID,
            SEX_IDENT_CD,
            BENE_DEATH_DT,
            FLOOR(DATEDIFF(day, CONVERT(date, BENE_BIRTH_DT, 106), '2021-12-31') / 365.25) AS age
        FROM beneficiary
        WHERE BENE_DEATH_DT IS NULL
    )
    SELECT
        measure,
        COUNT(*) AS eligible_population
    FROM (
        SELECT 'COL - Colorectal Cancer Screening (45-75)' AS measure FROM aged WHERE age BETWEEN 45 AND 75
        UNION ALL
        SELECT 'BCS - Breast Cancer Screening (F, 52-74)'  FROM aged WHERE age BETWEEN 52 AND 74 AND SEX_IDENT_CD = '2'
        UNION ALL
        SELECT 'CBP - Controlling High Blood Pressure (18-85)' FROM aged WHERE age BETWEEN 18 AND 85
        UNION ALL
        SELECT 'CDC - Diabetes Care (18-75)'               FROM aged WHERE age BETWEEN 18 AND 75
        UNION ALL
        SELECT 'AAB - Avoidance Antibiotics Bronchitis (18+)' FROM aged WHERE age >= 18
        UNION ALL
        SELECT 'AMR - Asthma Medication Ratio (5-64)'      FROM aged WHERE age BETWEEN 5 AND 64
        UNION ALL
        SELECT 'FUH - Follow-Up After Hospitalization (6+)' FROM aged WHERE age >= 6
        UNION ALL
        SELECT 'URI - Upper Respiratory Infection (3 mo+)' FROM aged WHERE age >= 0
        UNION ALL
        SELECT 'PPC - Prenatal/Postpartum Care (F, 8-64)'  FROM aged WHERE age BETWEEN 8 AND 64 AND SEX_IDENT_CD = '2'
        UNION ALL
        SELECT 'ABA - Adult BMI Assessment (18-74)'        FROM aged WHERE age BETWEEN 18 AND 74
    ) m
    GROUP BY measure
    ORDER BY measure
""", conn)

,measure,eligible_population
0,AAB - Avoidance Antibiotics Bronchitis (18+),7817
1,ABA - Adult BMI Assessment (18-74),5958
2,AMR - Asthma Medication Ratio (5-64),3674
3,"BCS - Breast Cancer Screening (F, 52-74)",2071
4,CBP - Controlling High Blood Pressure (18-85),7226
5,CDC - Diabetes Care (18-75),6138
6,COL - Colorectal Cancer Screening (45-75),4860
7,FUH - Follow-Up After Hospitalization (6+),8123
8,"PPC - Prenatal/Postpartum Care (F, 8-64)",1737
9,URI - Upper Respiratory Infection (3 mo+),8246


### Query 2 — ICD-10 code coverage (condition-based measures)

Some measures require a member to have a specific diagnosis before they're counted in the denominator. CBP requires a hypertension diagnosis (I10, I11, etc.). CDC requires a diabetes diagnosis (E10, E11). You can't be in the CBP denominator if you've never been diagnosed with the condition.

This query pulls all diagnosis codes from inpatient, outpatient, and carrier claims in 2021 and counts distinct members who have the relevant codes. It tells us whether the conditions are actually represented in the data — synthetic data doesn't always behave like real data, so we verify before building anything.

In [4]:
pd.read_sql("""
    WITH claims_2021 AS (
        -- Inpatient diagnoses 2021
        SELECT BENE_ID, PRNCPAL_DGNS_CD AS dx FROM inpatient
        WHERE CONVERT(date, CLM_FROM_DT, 106) BETWEEN '2021-01-01' AND '2021-12-31'
        UNION ALL
        SELECT BENE_ID, ICD_DGNS_CD1 FROM inpatient WHERE CONVERT(date, CLM_FROM_DT, 106) BETWEEN '2021-01-01' AND '2021-12-31'
        UNION ALL
        -- Outpatient diagnoses 2021
        SELECT BENE_ID, PRNCPAL_DGNS_CD FROM outpatient WHERE CONVERT(date, CLM_FROM_DT, 106) BETWEEN '2021-01-01' AND '2021-12-31'
        UNION ALL
        SELECT BENE_ID, ICD_DGNS_CD1 FROM outpatient WHERE CONVERT(date, CLM_FROM_DT, 106) BETWEEN '2021-01-01' AND '2021-12-31'
        UNION ALL
        -- Carrier diagnoses 2021
        SELECT BENE_ID, LINE_ICD_DGNS_CD FROM carrier WHERE CONVERT(date, CLM_FROM_DT, 106) BETWEEN '2021-01-01' AND '2021-12-31'
    )
    SELECT
        measure,
        COUNT(DISTINCT BENE_ID) AS members_with_dx
    FROM (
        SELECT 'CBP - Hypertension (I10, I11, I12, I13)' AS measure, BENE_ID FROM claims_2021
            WHERE dx LIKE 'I10%' OR dx LIKE 'I11%' OR dx LIKE 'I12%' OR dx LIKE 'I13%'
        UNION ALL
        SELECT 'CDC - Diabetes (E10, E11)', BENE_ID FROM claims_2021
            WHERE dx LIKE 'E10%' OR dx LIKE 'E11%'
        UNION ALL
        SELECT 'AAB - Acute Bronchitis (J20)', BENE_ID FROM claims_2021
            WHERE dx LIKE 'J20%'
        UNION ALL
        SELECT 'AMR - Asthma (J45)', BENE_ID FROM claims_2021
            WHERE dx LIKE 'J45%'
        UNION ALL
        SELECT 'FUH - Mental Illness (F20-F99)', BENE_ID FROM claims_2021
            WHERE dx LIKE 'F2%' OR dx LIKE 'F3%' OR dx LIKE 'F4%' OR dx LIKE 'F5%'
                  OR dx LIKE 'F6%' OR dx LIKE 'F7%' OR dx LIKE 'F8%' OR dx LIKE 'F9%'
        UNION ALL
        SELECT 'URI - Upper Respiratory (J00, J06)', BENE_ID FROM claims_2021
            WHERE dx LIKE 'J00%' OR dx LIKE 'J06%'
    ) coded
    GROUP BY measure
    ORDER BY measure
""", conn)

,measure,members_with_dx
0,AAB - Acute Bronchitis (J20),871
1,AMR - Asthma (J45),333
2,"CBP - Hypertension (I10, I11, I12, I13)",575
3,"CDC - Diabetes (E10, E11)",716
4,FUH - Mental Illness (F20-F99),298
5,"URI - Upper Respiratory (J00, J06)",46


### Query 3 — HCPCS/CPT code coverage (procedure-based measures)

Some measures are about whether a specific service happened — a mammogram, a colonoscopy, an office visit. Those services show up as HCPCS or CPT codes on carrier and outpatient claims.

This query checks whether those codes actually appear in 2021 claims and counts how many members have them. If a procedure code is absent from the data entirely, the measure can't be implemented — the numerator would always be zero and the rate would be meaningless.

In [5]:
pd.read_sql("""
    WITH hcpcs_2021 AS (
        SELECT BENE_ID, HCPCS_CD FROM carrier
        WHERE CONVERT(date, CLM_FROM_DT, 106) BETWEEN '2021-01-01' AND '2021-12-31'
          AND HCPCS_CD IS NOT NULL
        UNION ALL
        SELECT BENE_ID, HCPCS_CD FROM outpatient
        WHERE CONVERT(date, CLM_FROM_DT, 106) BETWEEN '2021-01-01' AND '2021-12-31'
          AND HCPCS_CD IS NOT NULL
    )
    SELECT
        measure,
        COUNT(DISTINCT BENE_ID) AS members_with_code
    FROM (
        SELECT 'COL - Colonoscopy/FOBT (45378,45380,45385,G0328,82274,82270)' AS measure, BENE_ID
            FROM hcpcs_2021
            WHERE HCPCS_CD IN ('45378','45380','45385','G0328','82274','82270')
        UNION ALL
        SELECT 'BCS - Mammography (77067,77066,77065)', BENE_ID
            FROM hcpcs_2021
            WHERE HCPCS_CD IN ('77067','77066','77065')
        UNION ALL
        SELECT 'CDC - HbA1c Test (83036,83037)', BENE_ID
            FROM hcpcs_2021
            WHERE HCPCS_CD IN ('83036','83037')
        UNION ALL
        SELECT 'PPC - Delivery codes (59400,59510,59610,59618)', BENE_ID
            FROM hcpcs_2021
            WHERE HCPCS_CD IN ('59400','59510','59610','59618')
        UNION ALL
        SELECT 'ABA - Office visits (99385-99387,99395-99397,99201-99215)', BENE_ID
            FROM hcpcs_2021
            WHERE HCPCS_CD IN (
                '99385','99386','99387','99395','99396','99397',
                '99201','99202','99203','99204','99205',
                '99211','99212','99213','99214','99215'
            )
    ) coded
    GROUP BY measure
    ORDER BY measure
""", conn)

,measure,members_with_code
0,"ABA - Office visits (99385-99387,99395-99397,9...",453
1,"BCS - Mammography (77067,77066,77065)",7
2,"COL - Colonoscopy/FOBT (45378,45380,45385,G032...",783


---
### Phase 5 — Coverage Analysis Conclusions

#### Keep — sufficient population and code coverage
| Measure | Eligible population | Members with codes/dx |
|---|---|---|
| COL — Colorectal Cancer Screening | 4,860 | 783 with colonoscopy/FOBT codes |
| AAB — Avoidance of Antibiotics for Bronchitis | 7,817 | 871 with bronchitis dx |
| ABA — Adult BMI Assessment | 5,958 | 453 with office visit codes |
| AMR — Asthma Medication Ratio | 3,674 | 333 with asthma dx |
| FUH — Follow-Up After Hospitalization | 8,123 | 298 with mental illness dx |

#### Drop — data not sufficient to implement
| Measure | Issue |
|---|---|
| BCS — Breast Cancer Screening | 2,071 eligible but only 7 members with mammography codes — numerator effectively zero |
| PPC — Prenatal and Postpartum Care | 0 members with delivery codes — cannot implement |
| URI — Upper Respiratory Infection | Only 46 members with URI diagnosis — too thin to produce meaningful rate |
| CBP — Controlling High Blood Pressure | Measuring BP control requires actual BP readings — clinical data not present in claims |
| CDC — Diabetes Care, HbA1c Testing | HbA1c test codes (83036, 83037) absent from carrier and outpatient across all years — lab claims not included in this dataset |

---
## Phase 5 — Replacement Measure Coverage Checks
Five replacement candidates for the five dropped measures (CBP, CDC, BCS, PPC, URI).
All five are purely claims-based — no EHR or lab data required.

If all five pass coverage checks, the final measure set will be:
COL, AAB, ABA, AMR, FUH, PCR, LBP, SAA, FUM, IET.

### PCR — Plan All-Cause Readmissions

**What it is:** A utilization measure that tracks how often patients are readmitted to the hospital within 30 days of a prior discharge.

**What it measures:**
- Denominator: All inpatient discharges in the measurement year (index admissions), excluding planned readmissions and transfers
- Numerator: Index admissions followed by an unplanned inpatient readmission within 30 days
- Direction: Lower is better — a high readmission rate signals poor care transitions or premature discharge

**Why it matters:** Readmission reduction has been a major focus of CMS payment policy since the Hospital Readmissions Reduction Program (HRRP) in 2012. It reflects the quality of care coordination at discharge and in the community.

**Data required:** Inpatient claims only. Uses CLM_FROM_DT (admission date) and CLM_THRU_DT (discharge date) to identify index admissions and calculate the 30-day window.

**Coverage check:** Count of 2021 inpatient index admissions (one row per claim, not per line), and count of those followed by a readmission within 30 days.

In [6]:
pd.read_sql("""
    WITH index_admissions AS (
        -- One row per inpatient claim in 2021 (CLM_LINE_NUM = 1 avoids counting each line separately)
        SELECT
            BENE_ID,
            CONVERT(date, CLM_FROM_DT, 106) AS admit_dt,
            CONVERT(date, CLM_THRU_DT, 106) AS discharge_dt
        FROM inpatient
        WHERE CLM_LINE_NUM = 1
          AND CONVERT(date, CLM_FROM_DT, 106) BETWEEN '2021-01-01' AND '2021-12-31'
    ),
    readmissions AS (
        -- For each index admission, look for any inpatient admission within 30 days of discharge
        SELECT DISTINCT i.BENE_ID, i.admit_dt
        FROM index_admissions i
        JOIN inpatient r
            ON i.BENE_ID = r.BENE_ID
            AND r.CLM_LINE_NUM = 1
            AND CONVERT(date, r.CLM_FROM_DT, 106) > i.discharge_dt
            AND CONVERT(date, r.CLM_FROM_DT, 106) <= DATEADD(day, 30, i.discharge_dt)
    )
    SELECT
        COUNT(*) AS index_admissions_2021,
        (SELECT COUNT(*) FROM readmissions) AS readmissions_within_30_days,
        ROUND(
            CAST((SELECT COUNT(*) FROM readmissions) AS FLOAT) / COUNT(*) * 100, 1
        ) AS readmission_rate_pct
    FROM index_admissions
""", conn)

,index_admissions_2021,readmissions_within_30_days,readmission_rate_pct
0,3049,955,31.3


### LBP — Use of Imaging Studies for Low Back Pain

**What it is:** An overuse measure that tracks whether members with acute low back pain received unnecessary imaging (X-ray, MRI, or CT scan) within 28 days of diagnosis.

**What it measures:**
- Denominator: Members aged 18-50 with a new acute low back pain diagnosis (ICD-10: M54.x) who had no prior back pain in the 6 months before the episode
- Numerator: Those who received a lumbar imaging study (X-ray, MRI, or CT) within 28 days
- Direction: Lower is better — clinical guidelines recommend against routine imaging for uncomplicated acute low back pain, as it rarely changes treatment and exposes patients to unnecessary radiation

**Why it matters:** Low back pain is one of the most common reasons for physician visits. Inappropriate imaging drives up cost without improving outcomes. This measure reflects adherence to evidence-based guidelines.

**Data required:** Diagnosis codes from outpatient and carrier claims to identify the episode; HCPCS codes from carrier and outpatient to identify imaging. Age from beneficiary.

**Imaging codes checked:**
- X-ray (lumbar spine): 72100, 72102, 72110, 72114, 72120
- MRI (lumbar spine): 72141, 72142, 72146, 72147, 72148, 72156, 72157, 72158
- CT (lumbar spine): 72125, 72126, 72127, 72128, 72129, 72130, 72131, 72132

**Coverage check:** Members aged 18-50 with M54 diagnosis in 2021, and separately how many of those had lumbar imaging codes in the same period.

In [7]:
pd.read_sql("""
    WITH aged AS (
        SELECT
            BENE_ID,
            FLOOR(DATEDIFF(day, CONVERT(date, BENE_BIRTH_DT, 106), '2021-12-31') / 365.25) AS age
        FROM beneficiary
        WHERE BENE_DEATH_DT IS NULL
    ),
    lbp_members AS (
        -- Members aged 18-50 with an M54 (low back pain) diagnosis in 2021
        SELECT DISTINCT d.BENE_ID
        FROM (
            SELECT BENE_ID, PRNCPAL_DGNS_CD AS dx FROM outpatient
                WHERE CONVERT(date, CLM_FROM_DT, 106) BETWEEN '2021-01-01' AND '2021-12-31'
            UNION ALL
            SELECT BENE_ID, ICD_DGNS_CD1 FROM outpatient
                WHERE CONVERT(date, CLM_FROM_DT, 106) BETWEEN '2021-01-01' AND '2021-12-31'
            UNION ALL
            SELECT BENE_ID, LINE_ICD_DGNS_CD FROM carrier
                WHERE CONVERT(date, CLM_FROM_DT, 106) BETWEEN '2021-01-01' AND '2021-12-31'
        ) d
        JOIN aged a ON d.BENE_ID = a.BENE_ID
        WHERE d.dx LIKE 'M54%'
          AND a.age BETWEEN 18 AND 50
    ),
    imaging_members AS (
        -- Members with lumbar imaging codes in 2021
        SELECT DISTINCT BENE_ID
        FROM (
            SELECT BENE_ID, HCPCS_CD FROM carrier
                WHERE CONVERT(date, CLM_FROM_DT, 106) BETWEEN '2021-01-01' AND '2021-12-31'
            UNION ALL
            SELECT BENE_ID, HCPCS_CD FROM outpatient
                WHERE CONVERT(date, CLM_FROM_DT, 106) BETWEEN '2021-01-01' AND '2021-12-31'
        ) h
        WHERE HCPCS_CD IN (
            '72100','72102','72110','72114','72120',       -- X-ray
            '72141','72142','72146','72147','72148',       -- MRI
            '72156','72157','72158',                       -- MRI with contrast
            '72125','72126','72127','72128','72129',       -- CT
            '72130','72131','72132'                        -- CT continued
        )
    )
    SELECT
        (SELECT COUNT(*) FROM lbp_members) AS members_with_lbp_dx,
        (SELECT COUNT(*) FROM imaging_members WHERE BENE_ID IN (SELECT BENE_ID FROM lbp_members)) AS lbp_members_with_imaging
""", conn)

,members_with_lbp_dx,lbp_members_with_imaging
0,124,0


### SAA — Adherence to Antipsychotic Medications for Individuals with Schizophrenia

**What it is:** A pharmacy adherence measure that tracks whether members with schizophrenia stay on their antipsychotic medications throughout the year.

**What it measures:**
- Denominator: Members aged 18-64 with schizophrenia (ICD-10: F20-F29) who had at least two encounters and at least one antipsychotic prescription during the measurement year
- Numerator: Members with a Proportion of Days Covered (PDC) ≥ 80% for antipsychotic medications
- Direction: Higher is better — consistent medication adherence is critical for managing schizophrenia and preventing relapse and hospitalization

**Why it matters:** Schizophrenia is a severe, chronic condition where medication non-adherence is the leading cause of relapse and rehospitalization. PDC ≥ 80% is the standard threshold used across adherence measures.

**Data required:** ICD-10 diagnosis codes (F20-F29) from inpatient/outpatient/carrier to identify the population; PDE (pharmacy) records to calculate days of medication supply and PDC. This is the only measure in our set that directly uses the Part D Event (PDE) table.

**Coverage check:** Members with schizophrenia diagnosis in 2021, and how many of those members also have PDE (pharmacy) records — confirming we can link the two data sources.

In [8]:
pd.read_sql("""
    WITH aged AS (
        SELECT
            BENE_ID,
            FLOOR(DATEDIFF(day, CONVERT(date, BENE_BIRTH_DT, 106), '2021-12-31') / 365.25) AS age
        FROM beneficiary
        WHERE BENE_DEATH_DT IS NULL
    ),
    schizophrenia_members AS (
        -- Members aged 18-64 with a schizophrenia-spectrum diagnosis (F20-F29) in 2021
        SELECT DISTINCT d.BENE_ID
        FROM (
            SELECT BENE_ID, PRNCPAL_DGNS_CD AS dx FROM inpatient
                WHERE CONVERT(date, CLM_FROM_DT, 106) BETWEEN '2021-01-01' AND '2021-12-31'
            UNION ALL
            SELECT BENE_ID, PRNCPAL_DGNS_CD FROM outpatient
                WHERE CONVERT(date, CLM_FROM_DT, 106) BETWEEN '2021-01-01' AND '2021-12-31'
            UNION ALL
            SELECT BENE_ID, LINE_ICD_DGNS_CD FROM carrier
                WHERE CONVERT(date, CLM_FROM_DT, 106) BETWEEN '2021-01-01' AND '2021-12-31'
        ) d
        JOIN aged a ON d.BENE_ID = a.BENE_ID
        WHERE d.dx LIKE 'F2%'
          AND a.age BETWEEN 18 AND 64
    )
    SELECT
        (SELECT COUNT(*) FROM schizophrenia_members) AS members_with_schizophrenia_dx,
        (
            SELECT COUNT(DISTINCT s.BENE_ID)
            FROM schizophrenia_members s
            JOIN pde p ON s.BENE_ID = p.BENE_ID
            WHERE CONVERT(date, p.SRVC_DT, 106) BETWEEN '2021-01-01' AND '2021-12-31'
        ) AS schizophrenia_members_with_pde_2021
""", conn)

,members_with_schizophrenia_dx,schizophrenia_members_with_pde_2021
0,0,0


### FUM — Follow-Up After Emergency Department Visit for Mental Illness

**What it is:** A follow-up measure that tracks whether members who visit the emergency department for a mental health condition receive outpatient follow-up care within 7 days.

**What it measures:**
- Denominator: Members with an ED visit with a principal mental illness diagnosis (ICD-10: F20-F99) who were not admitted inpatient from the ED
- Numerator: Those who had an outpatient mental health visit within 7 days of the ED discharge
- Direction: Higher is better — timely follow-up after an ED mental health crisis reduces the likelihood of repeat ED visits and hospitalization

**Why it matters:** ED visits for mental illness often represent a crisis point. Without timely follow-up, patients are at high risk of deterioration and return visits. This measure is a natural companion to FUH (Follow-Up After Hospitalization for Mental Illness), which we are already implementing.

**Data required:** Outpatient claims to identify ED visits (revenue center codes 0450-0459, 0981); carrier and outpatient claims to identify follow-up visits. Mental illness diagnosis codes (F20-F99).

**Coverage check:** Members with an outpatient claim in 2021 with a mental illness diagnosis — proxy for ED mental health visits, confirming the population exists in the data.

In [9]:
pd.read_sql("""
    WITH mental_health_outpatient AS (
        -- Members with a mental illness diagnosis on an outpatient claim in 2021
        -- Outpatient claims represent facility-based visits including ED visits
        SELECT DISTINCT BENE_ID
        FROM (
            SELECT BENE_ID, PRNCPAL_DGNS_CD AS dx FROM outpatient
                WHERE CONVERT(date, CLM_FROM_DT, 106) BETWEEN '2021-01-01' AND '2021-12-31'
            UNION ALL
            SELECT BENE_ID, ICD_DGNS_CD1 FROM outpatient
                WHERE CONVERT(date, CLM_FROM_DT, 106) BETWEEN '2021-01-01' AND '2021-12-31'
        ) d
        WHERE dx LIKE 'F2%' OR dx LIKE 'F3%' OR dx LIKE 'F4%' OR dx LIKE 'F5%'
           OR dx LIKE 'F6%' OR dx LIKE 'F7%' OR dx LIKE 'F8%' OR dx LIKE 'F9%'
    ),
    mental_health_carrier AS (
        -- Members with a mental illness diagnosis on a carrier claim in 2021
        -- Carrier claims capture follow-up outpatient visits with physicians
        SELECT DISTINCT BENE_ID
        FROM carrier
        WHERE CONVERT(date, CLM_FROM_DT, 106) BETWEEN '2021-01-01' AND '2021-12-31'
          AND (LINE_ICD_DGNS_CD LIKE 'F2%' OR LINE_ICD_DGNS_CD LIKE 'F3%'
            OR LINE_ICD_DGNS_CD LIKE 'F4%' OR LINE_ICD_DGNS_CD LIKE 'F5%'
            OR LINE_ICD_DGNS_CD LIKE 'F6%' OR LINE_ICD_DGNS_CD LIKE 'F7%'
            OR LINE_ICD_DGNS_CD LIKE 'F8%' OR LINE_ICD_DGNS_CD LIKE 'F9%')
    )
    SELECT
        (SELECT COUNT(*) FROM mental_health_outpatient) AS members_with_mh_outpatient_2021,
        (SELECT COUNT(*) FROM mental_health_carrier)    AS members_with_mh_carrier_2021
""", conn)

,members_with_mh_outpatient_2021,members_with_mh_carrier_2021
0,205,102


### IET — Initiation and Engagement of Substance Use Disorder Treatment

**What it is:** A two-part measure that tracks whether members diagnosed with a new substance use disorder (SUD) begin and then engage in treatment.

**What it measures:**
- Denominator: Members 13 and older with a new SUD diagnosis (ICD-10: F10-F19 — alcohol, opioid, cannabis, stimulant, sedative, and other substance disorders) with no SUD treatment in the prior 60 days
- Numerator Part 1 (Initiation): Members who receive SUD treatment within 14 days of the diagnosis
- Numerator Part 2 (Engagement): Members who receive two or more additional SUD treatment services within 34 days of initiation
- Direction: Higher is better for both parts

**Why it matters:** Substance use disorder is one of the most undertreated conditions in healthcare. The gap between diagnosis and treatment initiation is a major quality problem. IET directly measures whether the healthcare system is connecting people with treatment at the critical moment of diagnosis.

**Data required:** ICD-10 diagnosis codes (F10-F19) from any claim type to identify new SUD episodes; procedure codes from carrier and outpatient claims to identify SUD treatment services (H0001, H0004, H0005, H0007, H0015, H0020, H2035, T1006, 90832, 90834, 90837, 98960, 99202-99215 with SUD dx, among others).

**Coverage check:** Members with a new SUD diagnosis (F10-F19) in 2021, and how many of those members had any subsequent outpatient or carrier claim within 34 days — confirming treatment follow-up is detectable in the data.

In [10]:
pd.read_sql("""
    WITH sud_diagnosis AS (
        -- Members with a new SUD diagnosis (F10-F19) in 2021
        -- Using the earliest claim date as the index event
        SELECT BENE_ID, MIN(CONVERT(date, CLM_FROM_DT, 106)) AS first_sud_dt
        FROM (
            SELECT BENE_ID, CLM_FROM_DT, PRNCPAL_DGNS_CD AS dx FROM outpatient
                WHERE CONVERT(date, CLM_FROM_DT, 106) BETWEEN '2021-01-01' AND '2021-12-31'
            UNION ALL
            SELECT BENE_ID, CLM_FROM_DT, ICD_DGNS_CD1 FROM outpatient
                WHERE CONVERT(date, CLM_FROM_DT, 106) BETWEEN '2021-01-01' AND '2021-12-31'
            UNION ALL
            SELECT BENE_ID, CLM_FROM_DT, LINE_ICD_DGNS_CD FROM carrier
                WHERE CONVERT(date, CLM_FROM_DT, 106) BETWEEN '2021-01-01' AND '2021-12-31'
            UNION ALL
            SELECT BENE_ID, CLM_FROM_DT, PRNCPAL_DGNS_CD FROM inpatient
                WHERE CONVERT(date, CLM_FROM_DT, 106) BETWEEN '2021-01-01' AND '2021-12-31'
        ) d
        WHERE dx LIKE 'F1%'
        GROUP BY BENE_ID
    ),
    followup AS (
        -- Members with any subsequent claim within 34 days of their first SUD diagnosis
        SELECT DISTINCT s.BENE_ID
        FROM sud_diagnosis s
        JOIN (
            SELECT BENE_ID, CONVERT(date, CLM_FROM_DT, 106) AS svc_dt FROM outpatient
            UNION ALL
            SELECT BENE_ID, CONVERT(date, CLM_FROM_DT, 106) FROM carrier
        ) f ON s.BENE_ID = f.BENE_ID
        WHERE f.svc_dt > s.first_sud_dt
          AND f.svc_dt <= DATEADD(day, 34, s.first_sud_dt)
    )
    SELECT
        (SELECT COUNT(*) FROM sud_diagnosis) AS members_with_new_sud_dx_2021,
        (SELECT COUNT(*) FROM followup)      AS sud_members_with_followup_within_34_days
""", conn)

,members_with_new_sud_dx_2021,sud_members_with_followup_within_34_days
0,370,238


---
### Replacement Measure Coverage — Conclusions

#### Keep — sufficient data to implement
| Measure | Result |
|---|---|
| PCR — Plan All-Cause Readmissions | 3,049 index admissions in 2021; 955 readmissions within 30 days (31.3%) |
| FUM — Follow-Up After ED Visit for Mental Illness | 205 members with mental health outpatient claims; 102 with carrier claims |
| IET — Initiation and Engagement of SUD Treatment | 370 members with new SUD diagnosis; 238 with follow-up within 34 days |

#### Drop — data not sufficient to implement
| Measure | Issue |
|---|---|
| LBP — Use of Imaging for Low Back Pain | 124 members with low back pain diagnosis but 0 with lumbar imaging codes — procedure codes absent from dataset |
| SAA — Adherence to Antipsychotic Medications | 0 members with schizophrenia-spectrum diagnosis (F20-F29) — codes not present in synthetic dataset |

**Note on PCR rate:** A 31.3% readmission rate is higher than the real-world average (~15%). This is a synthetic data artifact and does not affect the implementability of the measure.

**Final measure set (8 measures):** COL, AAB, ABA, AMR, FUH, PCR, FUM, IET